# 03 - Detection
Run baseline spot detection on 20-50 images and save 5-10 overlay examples plus metrics.

In [ ]:
from pathlib import Path
import sys
import cv2
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_loader import load_metadata, assign_group_labels, get_channel_columns, load_multichannel_image
from preprocess import resize_image, normalize_image, clean_image
from detect import detect_spots, create_overlay
from features import compute_features

meta_path = PROJECT_ROOT / 'data' / 'raw' / 'metadata' / 'BBBC021_v1_image.csv'
images_dir = PROJECT_ROOT / 'data' / 'raw' / 'images'
overlay_dir = PROJECT_ROOT / 'outputs' / 'overlays'
metrics_dir = PROJECT_ROOT / 'outputs' / 'metrics'
overlay_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

meta = assign_group_labels(load_metadata(meta_path))
channel_cols = get_channel_columns(meta)
target_n = 40
sample_rows = []
for idx, row in meta.iterrows():
    try:
        _fused, _chs = load_multichannel_image(row, images_dir, channel_cols)
        sample_rows.append((idx, row))
    except Exception:
        continue
    if len(sample_rows) >= target_n:
        break

print('Detection sample size:', len(sample_rows))

In [ ]:
rows = []
for i, (idx, row) in enumerate(sample_rows):
    image_id = f'image_{idx:05d}'
    fused, _ = load_multichannel_image(row, images_dir, channel_cols)

    img = clean_image(normalize_image(resize_image(fused, (512, 512))))
    det = detect_spots(img, min_area=8, max_area=2000)

    feat = compute_features(
        image_gray=img,
        mask=det['mask'],
        spot_count=det['spot_count'],
        image_id=image_id,
        group=row['group'],
    )
    rows.append(feat)

    if i < 10:
        overlay = create_overlay(img, det['contours'])
        cv2.imwrite(str(overlay_dir / f'{image_id}_overlay.png'), overlay)

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(metrics_dir / 'image_metrics.csv', index=False)
metrics_df[['image_id', 'spot_count']].head(10).to_csv(metrics_dir / 'spot_count_sample.csv', index=False)

display(metrics_df[['image_id', 'spot_count']].head(10))
print('Saved overlays:', len(list(overlay_dir.glob('*_overlay.png'))))